In [3]:
%pip install --upgrade pandas numpy xgboost scikit-learn

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 18.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 16.8 MB/s  0:00:00 eta 0:00:01
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 23.9 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 25.3 MB/s  0:00:00 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [scikit-learn] [scikit-learn]
Not

In [4]:
import pandas as pd

In [5]:
demand = pd.read_excel('Dataset/PGCB_date_power_demand.xlsx')
demand.to_parquet('Dataset/PGCB_date_power_demand.parquet')

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
print(demand.dtypes)

In [ ]:
import numpy as np

# Ensure datetime is the correct data type
demand = demand.reset_index()
demand['datetime'] = pd.to_datetime(demand['datetime'])

# Set datetime as the index
demand = demand.set_index('datetime')


# If there are duplicate timestamps with conflicting values, average them for numeric columns and resolve 'remarks'

numeric_cols = demand.select_dtypes(include='number').columns
remarks_col = 'remarks'

def remark_resolver(x):
    unique_vals = x.dropna().unique()
    
    if len(unique_vals) == 0:
        return np.nan
    elif len(unique_vals) == 1:
        return unique_vals[0]
    else:
        return "Day_Peak"

if demand.index.duplicated().any():
    demand = demand.groupby(level=0).agg({
        **{col: 'mean' for col in numeric_cols},
        remarks_col: remark_resolver
    })
    
# Aligning the dataset to a perfect hourly grid. 
# Missing hours will be filled with NaN. We will NOT interpolate.
demand = demand.resample('H').asfreq()

window_size = 24

# Center=False is crucial here to prevent data leakage from the future into the past.
rolling_median = demand['demand_mw'].rolling(window=window_size, min_periods=12, center=False).median()

# Calculate Rolling IQR (Interquartile Range) to measure volatility robustly
rolling_q1 = demand['demand_mw'].rolling(window=window_size, min_periods=12, center=False).quantile(0.25)
rolling_q3 = demand['demand_mw'].rolling(window=window_size, min_periods=12, center=False).quantile(0.75)
rolling_iqr = rolling_q3 - rolling_q1

# Define dynamic bounds. 
# We use a multiplier of 3.0 (instead of the standard 1.5) to be conservative and avoid erasing real events.
iqr_multiplier = 3.0
upper_bound = rolling_median + (iqr_multiplier * rolling_iqr)
lower_bound = rolling_median - (iqr_multiplier * rolling_iqr)

# Flag points that fall outside these dynamic bounds
anomalies = (demand['demand_mw'] > upper_bound) | (demand['demand_mw'] < lower_bound)

# Replace flagged spikes with NaN so XGBoost can handle them organically
demand.loc[anomalies, 'demand_mw'] = np.nan

print(f"Total :{len(demand)}")
print(f"Anomalies : {anomalies.sum()}")
print(f"Total missing values : {demand['demand_mw'].isna().sum()}")

In [ ]:
weather = pd.read_excel('Dataset/weather_data.xlsx', skiprows=3)
weather.to_parquet('Dataset/weather_data.parquet')

In [ ]:
# Ensure datetime is the correct data type
weather = weather.reset_index()
weather['time'] = pd.to_datetime(weather['time'])

# Drop exact row duplicates
weather = weather.drop_duplicates()

# Set 'time' as the index to align with the demand dataset
weather = weather.set_index('time')

# If there are duplicate timestamps with conflicting values, average them
if weather.index.duplicated().any():
    weather = weather.groupby(level=0).mean()

# Resample to a strict hourly grid. Any missing hours will be introduced as NaN rows.
# This ensures the time axis perfectly mirrors the PGCB demand data.
weather = weather.resample('H').asfreq()

# Weather changes gradually. A missing temperature for 1 hour is safely inferred 
# by looking at the hour before and after. We use linear interpolation but strictly 
# limit it to 2 consecutive hours to prevent fabricating data over long outages.
weather = weather.interpolate(method='linear', limit=2)

print(f"Total : {len(weather)}")
print(f"Total missing values :\n{weather.isna().sum()}")

In [ ]:
demand['hour_of_day'] = demand.index.hour
demand['day_of_week'] = demand.index.dayofweek
demand['month'] = demand.index.month
demand['is_weekend'] = (demand['day_of_week'] >= 5).astype(int)

# The formula used is: sin(2 * pi * value / max_value) and cos(2 * pi * value / max_value)

# Hour of day cycle (Max 24)
demand['sin_hour'] = np.sin(2 * np.pi * demand['hour_of_day'] / 24)
demand['cos_hour'] = np.cos(2 * np.pi * demand['hour_of_day'] / 24)

# Day of week cycle (Max 7)
demand['sin_day_of_week'] = np.sin(2 * np.pi * demand['day_of_week'] / 7)
demand['cos_day_of_week'] = np.cos(2 * np.pi * demand['day_of_week'] / 7)

# Month cycle (Max 12)
demand['sin_month'] = np.sin(2 * np.pi * demand['month'] / 12)
demand['cos_month'] = np.cos(2 * np.pi * demand['month'] / 12)


In [ ]:
# Creating features that represent the demand at specific past intervals.
# t-1  t-2  t-24 t-168(week)
demand['demand_t-1'] = demand['demand_mw'].shift(1)
demand['demand_t-2'] = demand['demand_mw'].shift(2)
demand['demand_t-24'] = demand['demand_mw'].shift(24)
demand['demand_t-168'] = demand['demand_mw'].shift(168)

# Creating rolling statistics to capture recent trends and volatility.
windows = [3, 6, 24]

for w in windows:
    demand[f'rolling_mean_{w}h'] = demand['demand_mw'].rolling(window=w, min_periods=1).mean()
    demand[f'rolling_std_{w}h'] = demand['demand_mw'].rolling(window=w, min_periods=2).std()
    


In [ ]:
demand = demand.drop(columns=['level_0', 'index'], errors='ignore')
weather = weather.drop(columns=['level_0', 'index'], errors='ignore')

# Ensure the weather index has the same name as the demand index for a clean merge
weather.index.name = 'datetime'

# Perform a left join. 
# This keeps EVERY row in 'demand', and only brings in 'weather' rows 
# that have an exact matching timestamp. Extra weather timestamps are dropped.
data = demand.join(weather, how='left')

print(f"Original Demand shape: {demand.shape}")
print(f"Original Weather shape: {weather.shape}")
print(f"Combined Data shape:   {data.shape}")

# Check if the join introduced any new missing values in the weather columns
weather_cols = weather.columns
print("\nMissing values in attached weather data:")
print(data[weather_cols].isna().sum())

In [ ]:
print(demand.columns)

In [ ]:
print(weather.columns)

In [ ]:
print(data.columns)

In [ ]:
eco_indicators = pd.read_csv('Dataset/economic_indicators.csv')
eco_indicators.to_parquet('Dataset/economic_indicators.parquet')

In [ ]:

eco_clean = eco_indicators.drop(columns=['Country Name', 'Indicator Code'])

# Set 'Indicator Name' as the index and transpose so years become the row index
eco_t = eco_clean.set_index('Indicator Name').T

# Convert the year index from strings (e.g., '1960') to integers
eco_t.index = eco_t.index.astype(int)
eco_t.index.name = 'year'

# Clean up column names by replacing spaces and special characters with underscores
eco_t.columns = eco_t.columns.str.replace(r'[^a-zA-Z0-9]', '_', regex=True).str.strip('_')

# Instead of relying on .shift() (which breaks if a year is missing), 
# we physically add to the index. 
# For example, adding 1 to the index means the 2014 data now sits at index 2015, 
# meaning it will perfectly join to the 2015 hourly data.

eco_lag1 = eco_t.add_suffix('_Y_minus_1')
eco_lag1.index = eco_lag1.index + 1 

eco_lag2 = eco_t.add_suffix('_Y_minus_2')
eco_lag2.index = eco_lag2.index + 2

eco_lag5 = eco_t.add_suffix('_Y_minus_5')
eco_lag5.index = eco_lag5.index + 5

# Concatenate all the lagged features into a single lookup table
macro_features = pd.concat([eco_lag1, eco_lag2, eco_lag5], axis=1)


# Extract the year from the hourly datetime index to act as our join key
data['year'] = data.index.year

# Perform the left join using the 'year' column
data = data.join(macro_features, on='year', how='left')

# keeping the 'year' column to keep the data so that we can split

print(f"Final dataset shape: {data.shape}")
print("\nSample of appended macro columns:")
# Print the first few columns of the newly added macro data to verify
macro_cols = macro_features.columns[:3]
print(data[macro_cols].dropna().head())

In [ ]:
data = data.drop(columns=['day_of_week' , 'month', 'hour_of_day' , 'generation_mw', 'load_shedding', 'gas', 'liquid_fuel', 'coal', 'hydro', 'solar', 'wind', 'india_bheramara_hvdc', 'india_tripura', 'india_adani', 'nepal', 'remarks'])


In [ ]:
print(data.columns)

In [ ]:
X = data
y = data['demand_mw'].shift(-1)

X = X.iloc[:-1]
y = y.iloc[:-1]

print(f"{X.shape}")
print(f"{y.shape}")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_percentage_error


# Training Set: Everything up to and including 2023
train_data = data[data['year'] <= 2023].copy()

# Hold-out Test Set: Everything after 2023 (e.g., 2024 onwards)
test_data = data[data['year'] > 2023].copy()

# Drop the 'year' column from X
y_train = train_data['demand_mw'].shift(-1)
X_train = train_data.drop(columns=['year'])

y_test = test_data['demand_mw'].shift(-1)
X_test = test_data.drop(columns=['year'])

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")

# Seting Up Expanding-Window Cross Validation
# n_splits=5 will create 5 expanding chronological windows for validation
tscv = TimeSeriesSplit(n_splits=5)

# Defineing XGBoost Model and Hyperparameter Space
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist', random_state=93)

param_grid = {
    'n_estimators': [100, 300, 500],        # Number of trees
    'learning_rate': [0.01, 0.05, 0.1],     # Step size shrinkage
    'max_depth': [4, 6, 8],                 # Depth of each tree
    'subsample': [0.7, 0.8, 0.9],           # % of rows used per tree
    'colsample_bytree': [0.7, 0.8, 0.9]     # % of columns used per tree
}

# We use RandomizedSearchCV instead of GridSearchCV because hourly data 
# over multiple years is massive, and evaluating every single combination 
# would take far too long. n_iter=30 tests 30 random combinations.

# Use scikit-learn's built-in MAPE scorer (returns negative values for optimization)
mape_scorer = 'neg_mean_absolute_percentage_error'

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,                       # Increase this if you have more compute time
    scoring=mape_scorer,
    cv=tscv,                         # Pass the TimeSeriesSplit object here!
    verbose=2,                       # Prints progress
    n_jobs=-1,                       # Use all CPU cores
    random_state=93
)